<a href="https://colab.research.google.com/github/mirdbg/FinTok/blob/main/notebooks/02_audio/02_Audio_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinTok · Audio pipeline final

Genera **un WAV por escena** usando F5-TTS.

La decisión de quién habla ya viene del JSON producido por `src/text`.  
`src/audio` **no asigna voces por número de escena**.

Ejemplo:

```json
"speaker": "ANDREA"
```

selecciona la referencia de Andrea, mientras que:

```json
"speaker": "MIRIAM"
```

selecciona la referencia de Miriam.

El valor original de `speaker` se conserva en el JSON de salida.


## 1. Clonar o actualizar FinTok

Pon la URL real de tu repositorio en `REPO_URL`.

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/mirdbg/FinTok"
REPO_DIR = Path("/content/FinTok")

if (REPO_DIR / ".git").exists():
    print("✓ FinTok ya está clonado. Actualizando...")
    %cd /content/FinTok
    !git pull
    %cd /content
else:
    !git clone "$REPO_URL" /content/FinTok


Cloning into '/content/FinTok'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 193 (delta 64), reused 86 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 20.46 MiB | 10.08 MiB/s, done.
Resolving deltas: 100% (64/64), done.


## 2. Instalar dependencias

In [2]:
import os

# Evita el error de PYTHONHASHSEED observado en Colab.
os.environ["PYTHONHASHSEED"] = "0"

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U f5-tts soundfile

print("✓ Dependencias instaladas")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 61.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━

## 3. Comprobar referencias de voz

El pipeline espera estos WAV:

- Andrea: `data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav`
- Miriam: `data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav`


In [3]:
from pathlib import Path
from IPython.display import Audio, display

REPO_DIR = Path("/content/FinTok")

references = {
    "ANDREA": REPO_DIR / "data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav",
    "MIRIAM": REPO_DIR / "data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav",
}

for speaker, path in references.items():
    if not path.exists():
        raise FileNotFoundError(f"No encuentro {speaker}: {path}")

    print(f"✓ {speaker}: {path}")
    display(Audio(filename=str(path)))


✓ ANDREA: /content/FinTok/data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav


✓ MIRIAM: /content/FinTok/data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav


## 4. Seleccionar el JSON generado por texto

El JSON debe contener `speaker` en cada escena.

In [4]:
INPUT_JSON = "/content/FinTok/outputs/text/example_01.json"

print("JSON de entrada:", INPUT_JSON)


JSON de entrada: /content/FinTok/outputs/text/example_01.json


## 5. Revisar qué voz se usará

Esta comprobación se hace **antes de cargar F5-TTS**.

In [5]:
import json

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    preview = json.load(f)

for scene in preview["scenes"]:
    print(
        f"scene {scene['scene_id']} | "
        f"{scene['scene_type']} | "
        f"speaker={scene['speaker']}"
    )


scene 1 | hook | speaker=ANDREA
scene 2 | intro | speaker=MIRIAM
scene 3 | explanation | speaker=MIRIAM
scene 4 | outro | speaker=ANDREA


## 6. Generar los audios

El notebook llama al código real de `src/audio/audio_generator.py`.

In [6]:
import sys

sys.path.insert(0, "/content/FinTok")

from src.audio.audio_generator import generate_audio_from_json

result = generate_audio_from_json(
    input_json=INPUT_JSON,
    speed=1.0,
)

print("\n✓ Pipeline de audio terminado")


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Loading F5-TTS...
Download Vocos from huggingface charactr/vocos-mel-24khz


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 54.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

[3] explanation | speaker=MIRIAM | Apple Inc. states that its internal controls over financial reporting are designed to ensu
Converting audio...
Using cached preprocessed reference audio...
Using cached reference text...

ref_text   Welcome back! Stock prices are moving and investors are asking the same question. What happens next? Let's get started!. 
gen_text 0 Apple Inc. states that its internal controls over financial reporting are designed to ensure accurate financial statements. However, the company acknowledges inherent limitations – controls can’t always prevent or detect misstatements.
gen_text 1 Furthermore, projections of control effectiveness are subject to change due to evolving conditions, highlighting a potential vulnerability in their reporting process.


Generating audio in 2 batches...


100%|██████████| 2/2 [00:10<00:00,  5.11s/it]


[4] outro | speaker=ANDREA | That's your FinTok for today. See you in the next one.
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   Welcome back. Stock prices are moving and investors are asking the same question. What happens next? Let's get started. 
gen_text 0 That's your FinTok for today. See you in the next one.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:02<00:00,  2.63s/it]


✓ Generated 4 audio files
✓ Audio directory: /content/FinTok/data/audio_output/AAPL_2024
✓ Updated JSON: /content/FinTok/data/audio_output/AAPL_2024/script_with_audio.json

✓ Pipeline de audio terminado


## 7. Escuchar las cuatro escenas

In [7]:
from pathlib import Path
from IPython.display import Audio, display

repo_root = Path("/content/FinTok")

for scene in result["scenes"]:
    print(
        f"Escena {scene['scene_id']} | "
        f"{scene['scene_type']} | "
        f"{scene['speaker']}"
    )
    print(scene["text"])
    print("Audio:", scene["audio_path"])

    audio_path = Path(scene["audio_path"])
    if not audio_path.is_absolute():
        audio_path = repo_root / audio_path

    display(Audio(filename=str(audio_path)))
    print("-" * 80)


Escena 1 | hook | ANDREA
Did you know that Apple’s latest filing admits their internal controls aren’t foolproof, potentially leading to misstatements?
Audio: data/audio_output/AAPL_2024/scene_01_hook_andrea.wav


--------------------------------------------------------------------------------
Escena 2 | intro | MIRIAM
This is FinTok — your daily dose of financial knowledge.
Audio: data/audio_output/AAPL_2024/scene_02_intro_miriam.wav


--------------------------------------------------------------------------------
Escena 3 | explanation | MIRIAM
Apple Inc. states that its internal controls over financial reporting are designed to ensure accurate financial statements. However, the company acknowledges inherent limitations – controls can’t always prevent or detect misstatements. Furthermore, projections of control effectiveness are subject to change due to evolving conditions, highlighting a potential vulnerability in their reporting process.
Audio: data/audio_output/AAPL_2024/scene_03_explanation_miriam.wav


--------------------------------------------------------------------------------
Escena 4 | outro | ANDREA
That's your FinTok for today. See you in the next one.
Audio: data/audio_output/AAPL_2024/scene_04_outro_andrea.wav


--------------------------------------------------------------------------------


In [8]:
from pathlib import Path
import shutil

repo_root = Path("/content/FinTok")

# Carpeta de salida dentro del repositorio
output_dir = repo_root / "outputs" / "audio" / "example_01"
output_dir.mkdir(parents=True, exist_ok=True)

for scene in result["scenes"]:
    scene_id = int(scene["scene_id"])

    # Ruta del audio generado por el pipeline
    source_path = Path(scene["audio_path"])

    if not source_path.is_absolute():
        source_path = repo_root / source_path

    # Nombre final
    destination_path = output_dir / f"audio_{scene_id:02d}.wav"

    shutil.copy2(source_path, destination_path)

    print(f"✓ {destination_path}")

print("\n✓ Audios guardados en:")
print(output_dir)

✓ /content/FinTok/outputs/audio/example_01/audio_01.wav
✓ /content/FinTok/outputs/audio/example_01/audio_02.wav
✓ /content/FinTok/outputs/audio/example_01/audio_03.wav
✓ /content/FinTok/outputs/audio/example_01/audio_04.wav

✓ Audios guardados en:
/content/FinTok/outputs/audio/example_01


## 8. Ejecución fuera de Colab

Desde la raíz de FinTok:

```bash
python -m src.audio.audio_generator data/outputs/script.json
```

Opcionalmente:

```bash
python -m src.audio.audio_generator data/outputs/script.json \
    --output-dir data/audio_output/AAPL_2024 \
    --output-json data/audio_output/AAPL_2024/script_with_audio.json \
    --speed 1.0
```

El flujo queda:

`src/text → JSON con speaker → src/audio → 1 WAV por scene → JSON con audio_path`
